In [1]:
import heapq
INITIAL_STATE = (
    (2, 8, 3),
    (1, 6, 4),
    (7, 0, 5)
)

FINAL_STATE = (
    (1, 2, 3),
    (8, 0, 4),
    (7, 6, 5)
)

class Node:
    def __init__(self, state, parent=None, action=None, g=0, h=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f

def get_misplaced_tiles(state, goal_state):
    count = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0 and val != goal_state[r][c]:
                count += 1
    return count

def find_blank(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c

def get_successors(node, goal_state):
    r, c = find_blank(node.state)

    moves = [
        (0, -1, 'L'),
        (-1, 0, 'U'),
        (0, 1, 'R'),
        (1, 0, 'D')
    ]

    children = []
    for dr, dc, action in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = [list(row) for row in node.state]
            new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
            tuple_state = tuple(tuple(row) for row in new_state)
            g_child = node.g + 1
            h_child = get_misplaced_tiles(tuple_state, goal_state)

            children.append(Node(tuple_state, parent=node, action=action, g=g_child, h=h_child))

    return children

def reconstruct_path(node):
    path = []
    current = node
    while current:
        path.append(current)
        current = current.parent
    return path[::-1]

def a_star_search(initial_state, goal_state):


    h0 = get_misplaced_tiles(initial_state, goal_state)
    initial_node = Node(state=initial_state, g=0, h=h0)


    counter = 0
    frontier = []
    heapq.heappush(frontier, (initial_node.f, counter, initial_node))


    explored = set()

    while frontier:

        _, _, n = heapq.heappop(frontier)

        if n.state in explored:
            continue
        explored.add(n.state)


        if n.state == goal_state:
            return reconstruct_path(n)


        for child in get_successors(n, goal_state):
            if child.state not in explored:
                counter += 1

                heapq.heappush(frontier, (child.f, counter, child))

    return None

def print_solution(path):
    if not path:
        print("No solution found.")
        return

    print(f"Solution found in {len(path) - 1} steps:\n")
    for step, node in enumerate(path):
        print(f"Step {step} (Action: {node.action or 'Start'} | g={node.g}, h={node.h}, f={node.f}):")
        for row in node.state:
            print(" ", [val if val != 0 else ' ' for val in row])
        print()
if __name__ == "__main__":
    solution_path = a_star_search(INITIAL_STATE, FINAL_STATE)
    print_solution(solution_path)



Solution found in 5 steps:

Step 0 (Action: Start | g=0, h=4, f=4):
  [2, 8, 3]
  [1, 6, 4]
  [7, ' ', 5]

Step 1 (Action: U | g=1, h=3, f=4):
  [2, 8, 3]
  [1, ' ', 4]
  [7, 6, 5]

Step 2 (Action: U | g=2, h=3, f=5):
  [2, ' ', 3]
  [1, 8, 4]
  [7, 6, 5]

Step 3 (Action: L | g=3, h=2, f=5):
  [' ', 2, 3]
  [1, 8, 4]
  [7, 6, 5]

Step 4 (Action: D | g=4, h=1, f=5):
  [1, 2, 3]
  [' ', 8, 4]
  [7, 6, 5]

Step 5 (Action: R | g=5, h=0, f=5):
  [1, 2, 3]
  [8, ' ', 4]
  [7, 6, 5]

